# 原稿管理システム

## 設定

In [ ]:
# アンソロのフォルダ
folder_path = "/content/drive/MyDrive/~~~"
# 原稿のサイズ(px)を入力（横、縦）
manuscript_size = (3638, 5102)
# 実際の同人誌のサイズを入力（横、縦）
book_size = (3496, 4961)
# 本文モノクロ(=True)かフルカラー(=False)か
color_type = True
# 解像度
dpi = 600
# 右綴じ？左綴じ？
toji = "右綴じ"


# 以下、詳細設定

# 目次制作利用？TRUE or FALSE
mokuji_make = True
# 目次のBOX開始位置（左上の位置）
mokuji_box_position = (0, 0)
# ノンブル付利用？
has_nombre_book = True
# ノンブルのサイズ(倍率)、位置(横軸:{外・中}, 縦軸:{上・中・下}）、基本枠との間隔（横、縦）、原稿端明暗で色を変えるか、ノンブル色
# ノンブルを配置する場所が、明るい→黒のノンブル、暗い→白のノンブル、というように色を使い分けができます。
# 黒=(0, 0, 0), 赤＝(255, 0, 0)、のように指定。色使い分けがFalseのときは1色目のみが使用されます
nombre_setting = [1.0, ("外", "下"), (170, 170), True, [(0, 0, 0), (255, 255, 255)]]
# 他の提出物の名前（他に何かありましたら書いてください）
check_file_names = []

## 利用準備

最初に1回実行したら大丈夫です。
ライブラリのインポートが主です

In [ ]:
# マウント
from google.colab import drive
drive.mount('/content/drive')
%cd $folder_path

# GoogleColabからspreadsheetを操作するための認証
from google.colab import auth
auth.authenticate_user()

from PIL import Image, ImageFilter, ImageChops, ImageStat
import os
import pathlib

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

#使う関数の読み込み
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Tuple
import re
# !pip install natsort
from natsort import natsorted
import glob
import csv

!pip install img2pdf

import img2pdf
import logging

# global_pageの定義
global_page = 0


### 原稿1枚を管理するclass

In [ ]:
# ないページを補完する、白ページのクラス
@dataclass
class NonePage:
  author: str
  output_path: str | None = None
  page_global: int | None = None
  white_image = Image
  white_page = Image

  def __post_init__(self):
    if self.output_path:
      NonePage.white_image.save(self.output_path, dpi=(dpi, dpi))

  @classmethod
  def make_white_image(cls):
    if color_type:
      cls.white_image = Image.new("L", manuscript_size, color=255)
      cls.white_page = Image.new("L", book_size, color=255)
    else:
      cls.white_image= Image.new("RGB", manuscript_size, color=(255, 255, 255))
      cls.white_page = Image.new("RGB", book_size, color=(255, 255, 255))


# 原稿（1枚）のクラス
@dataclass
class ManuscriptPage:
    path: str# ファイル名
    is_file: bool = field(init=False) # そのファイルは存在するか
    author: str = field(init=False) # 執筆者の名前
    page_local: int = field(init=False) # その執筆者のコーナーの中で何ページ目か
    page_global: int | None = None # アンソロ全体で何ページ目か
    is_right: bool | None = None # ノンブル貼付箇所は明るい？暗い？
    has_nomburu: bool = field(init=False) # このページにノンブルを付けるか
    output_path: str | None = None # 出力するファイル名
    errors: list[str] = field(default_factory=list) # 不備の有無と不備の種類
    nombre_images: dict = field(default_factory=dict) # このページに張り付けるノンブル画像
    white_image = Image # 不備があったり、提出がないページはこれを利用する

    def __post_init__(self):
        # 渡されたpath(ファイル名)から、page_local(その執筆者の中で何ページ目か)や、author(執筆者名)を抽出
        dir_path, filename = os.path.split(self.path)
        author_dir = os.path.basename(dir_path)
        self.author = author_dir[:-1] if author_dir.endswith("様") else author_dir

        name, ext = os.path.splitext(filename)
        m = re.match(r"^(?P<base>.+?)\((?P<num>\d+)\)$", name)
        if m: # google driveの都合でファイル名の末尾に(1), (2)...が付いている可能性がある。それを除去
          name = m.group("base")
        self.has_nomburu = "n" in name

        page_part = name.replace("n", "") if self.has_nomburu else name
        try:
            self.page_local = int(page_part.replace("原稿", "").replace("_", ""))
        except ValueError:
            self.errors.append(f"[NAME NG] {filename} ERROR")
            self.page_local = None

        self.is_file = os.path.isfile(self.path)


    @classmethod
    def load_nombre_images(cls):
        # ノンブルの設定や、ノンブル画像を読み込み、ノンブルとして使えるようにする
        cls.nombre_images = {'h': None, 'w': []}
        scale = nombre_setting[0]
        adaptive = nombre_setting[3] # 明暗で色を変えるか(bool)
        colors = []
        for c in nombre_setting[4]:
          colors.append(c)
          if not adaptive:
            break

        # モノクロ本の場合、ノンブルカラーをモノクロに変換
        if color_type:
          for i, color in enumerate(colors):
            r, g, b = color
            y = (299 * r + 587 * g + 114 * b) // 1000
            colors[i] = 0 if y < 0 else 255 if y > 255 else int(y)


        # αを読み込み
        alpha_masks = []
        widths = []
        height = None

        for number in range(10):
            p = os.path.join(folder_path+"/原稿/その他/nombre_image", f"number_{number}.png")
            try:
              im = Image.open(p).convert("LA")
            except:
              print(f"ノンブル画像{number}が読み込めませんでした")
              alpha_masks.append(None)
              widths.append(None)
              global has_nombre_book
              has_nombre_book = False
              continue

            if scale != 1.0:
                im = im.resize(
                    (int(im.width * scale), int(im.height * scale)),
                    resample=Image.Resampling.NEAREST
                )

            l, a = im.split()
            alpha_masks.append(a)
            widths.append(im.width)
            if height is None:
                height = im.height

        cls.nombre_images['w'] = widths
        cls.nombre_images['h'] = height

        # 色ごとにglyphを生成（最大2色×10枚だけ）
        for i, color_final_value in enumerate(colors):
            key = f"colors{i}"
            cls.nombre_images[key] = []

            for a in alpha_masks:
              if a is None:
                cls.nombre_images[key].append(None)
                continue

              if color_type:
                  # モノクロ本：color_final_value は int（L値）
                  fill = Image.new("L", a.size, int(color_final_value))
                  cls.nombre_images[key].append(Image.merge("LA", (fill, a)))
              else:
                  # カラー本：color_final_value は (r,g,b)
                  fill = Image.new("RGB", a.size, (r, g, b))
                  cls.nombre_images[key].append(Image.merge("RGBA", (*fill.split(), a)))

        print("ノンブル画像　読み込み完了")


    @classmethod
    def make_white_image(cls):
      if color_type:
        cls.white_image = Image.new("L", manuscript_size, color=255)
      else:
        cls.white_image= Image.new("RGB", manuscript_size, color=(255, 255, 255))

    def is_right_page(self) -> bool:
      # 見開き時、右ページ(=True)か、左(=False)か
      if self.page_global== None:
        return None
      if toji == "右綴じ":
        return (self.page_global % 2) == 0
      elif toji == "左綴じ":
        return (self.page_global % 2) == 1
      else:
        return None


    # 保存した原稿画像を断ち切り線でトリミングして返す。なければ、Noneが返る
    def get_book_style(self):
      if not self.output_path:
        return None
      try:
        im = Image.open(self.output_path)
        return im.crop(((manuscript_size[0] - book_size[0]) // 2,
                        (manuscript_size[1] - book_size[1]) // 2,
                        (manuscript_size[0] + book_size[0]) // 2,
                        (manuscript_size[1] + book_size[1]) // 2))
      except:
          return None



    def _is_grayscale(self, im: Image.Image) -> bool:
      # αは無視する。LAは即OK。RGB/RGBA/P等は縮小して色差判定
      if im.mode in ("1", "L", "LA"):
        return True

      # パレット等も含めてRGB化して判定
      small = im.convert("RGB")
      small.thumbnail((256, 256))  # かなり軽くする

      r, g, b = small.split()
      if ImageChops.difference(r, g).getbbox() is not None:
        return False
      if ImageChops.difference(r, b).getbbox() is not None:
        return False
      return True


    def calc_nombre_position(self, total_w: int, digit_h: int,) -> tuple[int, int]:
      # ノンブルの貼り付け開始位置(左上）を計算する
      # nombre_setting = [scale, (横:外/中, 縦:上/中/下), (margin_x, margin_y)]
      _, (pos_x, pos_y), (mx, my), _, _ = nombre_setting

      W, H = manuscript_size

      # 横位置計算（外 or 中）
      if pos_x == "中":
        x = (W - total_w) // 2
      else:
        # 外：見開きで左右を変える
        is_right = self.is_right_page()
        if is_right:
            x =  W - mx - total_w
        else:
            x = mx

      # 縦位置計算（上/中/下）
      if pos_y == "上":
        y = my
      elif pos_y == "中":
        y = (H - digit_h) // 2
      else:
        y = H - my - digit_h

      return x, y

    @staticmethod
    def is_region_dark(base_img: Image.Image, box, threshold: int = 127) -> bool:
      # box = (x0, y0, x1, y1)
      # 画像の指定部分が暗い→返True

      region = base_img.crop(box).convert("L")
      mean = ImageStat.Stat(region).mean[0]
      return mean <= threshold


    def composite_nombre(self, img: Image.Image) -> Image.Image:
        str_num = str(self.page_global)

        # 幅計算（数字ごとに横幅が違う前提）
        digit_h = ManuscriptPage.nombre_images['h']
        w_list = ManuscriptPage.nombre_images["w"]
        total_w = sum(w_list[int(ch)] for ch in str_num)


        # 置き位置を計算
        x, y = self.calc_nombre_position(total_w=total_w, digit_h=digit_h)

        if nombre_setting[3]:  # 用いるノンブルの色を設定
          # 背景の明るさ判定（貼る予定領域）
          box = (x, y, x + total_w, y + digit_h)
          use_color1 = ManuscriptPage.is_region_dark(img, box) # 規定よりも暗かったらcolor1を用いる
          idx = 1 if use_color1 else 0
        else:
          idx = 0


        # 貼り付け
        glyphs = ManuscriptPage.nombre_images[f"colors{idx}"]
        cx = x
        for ch in str_num:
            d = glyphs[int(ch)]
            img.paste(d, (cx, y), mask=d)  # LAでもRGBAでもOK（αがマスクになる）
            cx += w_list[int(ch)]

        return img



    # ---- 原稿の画像ファイルを開いて、チェック、ノンブルつける、保存 ----
    def inspect_and_export(self, now_global_page:int) -> bool:
      # Return True→保存　False→警告を追加し、このページは保存しない
      errors = []  # このページのエラーを集める
      if not self.is_file:
        errors.append(f"[FILE None]{self.page_local}")
        return False

      with Image.open(self.path) as im:
        # カラーチェック（モノクロ本の場合のみ）
        if color_type:
            if not self._is_grayscale(im):
                errors.append(f"[COLOR NG] mode={im.mode} (monochrome book)")

        # サイズチェック
        if im.size != manuscript_size:
            errors.append(f"[SIZE NG] expected={manuscript_size} actual={im.size}")

        # 情報登録
        self.page_global = now_global_page
        self.is_right = self.is_right_page()
        self.output_path = folder_path + f"/原稿/全ての原稿/{self.page_global:03d}.png"

        # エラーが1つでもあれば、両方チェックした上でまとめて返す/白原稿を保存
        if errors:
          self.errors = errors
          self.output_path = folder_path + f"/原稿/全ての原稿/{self.page_global:03d}.png"
          ManuscriptPage.white_image.save(self.output_path, dpi=(dpi, dpi))
          return False

        # ノンブル合成（必要なときだけ）
        work = im
        if has_nombre_book and self.has_nomburu:
          work = self.composite_nombre(work)

        # 保存モード＆dpi
        if color_type:
            work_out = work.convert("L")
            work_out.save(self.output_path, quality=95, dpi=(dpi, dpi))
        else:
            work_out = work.convert("RGB")
            work_out.save(self.output_path, quality=95, dpi=(dpi, dpi))

        return True

### 執筆者のclass

In [ ]:

# ファイル名的に提出原稿ではないもの
@dataclass
class RejectedFile:
    path: str
    filename: str
    reason: str  # 基本は 「名前が異なります」が入る

# 執筆者のクラス
@dataclass
class Author:
  name: str # ペンネーム
  manuscript_type:str #漫画/イラスト/小説/その他
  expected_pages: int #予定ページ数）
  expected_start_page: int | None = None #予定スタートページ
  sequence: int | None = None # 掲載の順番
  folder: str | None = None # 提出BOXなければNone

  submit_pages: list = field(default_factory=list)# 提出された原稿ページ(int)たち。不備あり原稿含む
  manuscripts: dict = field(default_factory=dict)# 原稿画像であろうもの
  rejected: list[RejectedFile] = field(default_factory=list)#原稿でないもの
  white_page: NonePage | None = None # 原稿が奇数でなくて左右が揃わないとき、揃えるために作られる

  errors: dict = field(default_factory=dict)

  MANUSCRIPT_RE = re.compile(r"^原稿n?_?(?P<page>[0-9]+)n?(?:\((?P<dup>[0-9]+)\))?\.png$")
  # 許可: 原稿_001.png / 原稿n_001.png / 原稿n_001(1).png


  def __post_init__(self):
    if self.folder:
      if not os.path.isdir(self.folder):
        self.folder = None
    self.errors['author'] = []


  def set_folder(self, folder_probably):
    if os.path.isdir(folder_probably):
        self.folder = folder_probably


  def scan_folder(self):
    self.manuscripts = {}
    self.rejected = []
    self.submit_pages = []
    # 提出Boxから原稿らしきファイルを拾う
    manuscript_probably = natsorted([str(path) for path in list((pathlib.Path(self.folder)).glob('*.png'))])
    if len(manuscript_probably)==0:# 1枚も原稿ファイルが提出なかったら、まだ提出がないとしてFalse
      self.errors['author'].append(f"提出無し")
      return False

    for filename in manuscript_probably:
      filename_check = Author.MANUSCRIPT_RE.match(filename.split('/')[-1])
      if filename_check:
        self.submit_pages.append(int(filename_check.group('page')))
        self.manuscripts[int(filename_check.group('page'))] = ManuscriptPage(filename)
      else:
          self.rejected.append(RejectedFile(filename, filename.split('/')[-1], "ファイル名から原稿ではないと判断しました"))

    if len(self.submit_pages) == 0:
      self.errors['author'].append(f"提出無し")
    if max(self.submit_pages)>self.expected_pages:
      self.errors['author'].append(f"[PAGE OVER] max: {max(self.submit_pages)}")
    return True


  def manuscripts_check(self, start_page):
    if not self.submit_pages:
      return start_page
    print("原稿チェック開始", end="")
    now_global_page = start_page
    for local_num in range(0, max(self.submit_pages)+1):
      if local_num in self.submit_pages:
        manuscript = self.manuscripts[local_num]
        okmanu = manuscript.inspect_and_export(now_global_page)
        self.errors[manuscript.page_local] = manuscript.errors
        print(".", end="")
      else:
        none_white_page = NonePage(self.name, folder_path + f"/原稿/全ての原稿/{now_global_page:03d}.png", now_global_page)
      now_global_page+=1
    if now_global_page%2==1 and self.manuscript_type != "その他":
      self.white_page = NonePage(self.name, folder_path + f"/原稿/全ての原稿/{now_global_page:03d}.png", now_global_page)
      now_global_page += 1
      self.errors['author'].append(f"[PAGE ODD] add white page at the last")
    print("完了")
    return now_global_page


  # エラータグから説明用の文章を作る
  def to_polite(msg: str):
    msg = msg.strip()

    # タグ抽出
    m = re.match(r"^\[(?P<tag>[A-Z ]+)\]\s*(?P<body>.*)$", msg)
    tag = m.group("tag") if m else ""
    body = m.group("body") if m else msg

    # タグごとの説明文
    if tag == "PAGE OVER":
        return ["《！》ページ数確認", "登録されているページ数と提出内容に差があります。登録変更の旨を主催までお願いします"]
    if tag == "PAGE ODD":
        return ["《！》ページ数調整", "ページ数が偶数でした"]
    if tag == "NAME NG":
        return ["《！》ファイル名", "原稿のファイル名形式が規定と異なる場合に表示されます"]
    if tag == "COLOR NG":
        return ["《！》カラー原稿", "モノクロ以外の色が検出された場合に表示されます"]
    if tag == "SIZE NG":
        return ["《！》画像サイズ", f"原稿サイズが規定{manuscript_size}と一致しない場合に表示されます"]

    # 未知タグはそのまま
    return ["【確認】", f"{tag}{body}"]


  def export_report_csv(self):
    # self.folder に原稿チェック結果CSVを保存する
    if not self.folder or not self.submit_pages:
        return

    csv_file_name = f"{self.folder}/{self.name}様_原稿チェック結果.csv"

    with open(csv_file_name, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)

        # 情報
        w.writerow(["本結果は自動チェックによるものです。最終的な確認は、後ほど主催が目視で行います"])
        w.writerow([])
        w.writerow([f"{self.name}様", f"{self.manuscript_type}参加", f"予定：{self.expected_pages}ページ", " ", " "])

        # authorレベルのエラー（最大2行想定、あれば出す）
        author_msgs = self.errors.get("author", [])
        for m in author_msgs:
          w.writerow([" ".join(Author.to_polite(m))])

        w.writerow([])  # 空行

        # 各ページの提出状況
        w.writerow(["各ページの提出状況"])
        w.writerow(["ページ", "状況"])

        submit_pages = set(getattr(self, "submit_pages", []))

        # 表示する最大ページ
        max_submit = max(submit_pages) if submit_pages else 0
        max_page = max(max_submit, int(self.expected_pages))


        #原稿本体の不備確認結果を表示
        for p in range(0, max_page + 1):
            if p in submit_pages:
                page_errors = self.errors.get(p, [])
                if not page_errors:
                    w.writerow([p, "OK"])
                else:
                    manuscript_error_strs = [Author.to_polite(m) for m in page_errors]
                    w.writerow([p, " 　".join(m[0] for m in manuscript_error_strs)])
            else:
                w.writerow([p, "提出がありません"])

        w.writerow([])  # 空行
        w.writerow(["原稿のチェック項目一覧"])
        w.writerow(["タグ", "説明"])
        if color_type:
          w.writerow(Author.to_polite("[COLOR NG] body"))
        w.writerow(Author.to_polite("[SIZE NG] body"))
        w.writerow([])  # 空行
        w.writerow([])  # 空行

        # 3) 弾いたファイル一覧
        w.writerow(["以下はファイル名が規定形式と異なっていたため、原稿として認識されませんでした"])
        w.writerow(["原稿として提出したものの場合は、ファイル名をご確認ください"])
        w.writerow(["ファイル名"])
        if getattr(self, "rejected", None):
            for r in self.rejected:
                w.writerow([r.filename])
        else:
            w.writerow(["（なし）", ""])



  # 原稿を断ち切り線でトリミングし、見開きで繋げる。ページがない部分は白ページ
  def make_mihiraki(self, page_front_num:int, page_back_num:int):
    if color_type:
      dodai = Image.new("L", (book_size[0]*2, book_size[1]), color=(255))
    else:
      dodai = Image.new("RGB", (book_size[0]*2, book_size[1]), color=(255, 255, 255))
    if page_front_num in self.submit_pages:
        page_front = self.manuscripts[page_front_num].get_book_style()
        if page_front:
          dodai.paste(page_front, (book_size[0] if toji=="右綴じ" else 0, 0))
    if page_back_num in self.submit_pages:
        page_back = self.manuscripts[page_back_num].get_book_style()
        if page_back:
          dodai.paste(page_back, (0 if toji=="右綴じ" else book_size[0], 0))
    return dodai

  def preview_for_author(self):
    # 執筆者向け　見開きプレビューを作る
    if not self.submit_pages or self.manuscript_type=="その他" or not self.folder:
        return False

    if not os.path.exists(f"{self.folder}/プレビュー"):
      os.mkdir(f"{self.folder}/プレビュー")
    print("プレビュー作成開始", end="")
    for local_page in range(0, max(self.submit_pages)+1, 2):
      preview_img = self.make_mihiraki(page_front_num=local_page, page_back_num=local_page+1)
      preview_img.save(f"{self.folder}/プレビュー/preview_{local_page}_{local_page+1}.png", dpi=(dpi, dpi))
      print(".", end="")
    print("完了")


  def print_status_author(self):
    print(f"名前: {self.name}  参加形態: {self.manuscript_type}  予定ページ数: {self.expected_pages}  掲載: {self.sequence}番目")
    for error in self.errors['author']:
      print(f"※ {error}")
    if self.submit_pages:
      for i in range(max(max(self.submit_pages), self.expected_pages)+1):
        if i in self.submit_pages:
          if self.manuscripts[i].errors:
            m = [re.match(r"^\[(?P<tag>[A-Z ]+)\]\s*(?P<body>.*)$", msg) for msg in self.errors[i]]
            tag = [f"[{msg.group("tag")}]" if msg else "" for msg in m]
            print(f"{i} > {self.manuscripts[i].page_global:03d} : {", ".join(tag)}")
          else:
            print(f"{i} > {self.manuscripts[i].page_global:03d} : OK")
        else:
          print(f"{i} >     : None")
      if self.white_page:
        print(f"  > {self.white_page.page_global:03d} : white page")



  def author_check(self, start_global_page):
    if not self.folder:
      return start_global_page
    self.errors = {}
    self.errors['author'] = []
    self.white_page = None
    self.scan_folder()

    end_global_page = self.manuscripts_check(start_global_page)
    self.print_status_author()
    self.export_report_csv()
    self.preview_for_author()
    print("-"*30)
    return end_global_page

### 目次を作る関数
この部分に関しては各自書き換えて使ってください

In [ ]:

table_numbers = []
table_number_widths = []
table_numbers_mini = []
table_number_mini_widths = []

def table_reset():
  global mokuji_make, table_numbers, table_number_widths, table_numbers_mini, table_number_mini_widths
  if not mokuji_make:
    print("現在、目次の作成機能がOFFになっています")
    return False

  try:
    base_img1 = Image.open("原稿/目次/素材/土台1.png")
    base_img2 = Image.open("原稿/目次/素材/土台2.png")
  except:
    mokuji_make = False
    return False

  table_numbers = []
  table_number_widths = []
  height = None

  for number in range(10):
      p = os.path.join(folder_path+"/原稿/目次/素材/", f"{number}.png")
      try:
        im = Image.open(p)
      except:
        print(f"目次用ノンブル画像{number}が読み込めませんでした")
        table_numbers.append(None)
        table_number_widths.append(None)
        mokuji_make = False
        continue

      table_numbers.append(im)
      if number == 7:
        table_number_widths.append(im.width-10)
      else:
        table_number_widths.append(im.width)

      try:
        im = Image.open(f"{folder_path}/原稿/目次/素材/2{number}.png")
      except:
        None
      table_numbers_mini.append(im)
      table_number_mini_widths.append(im.width)


  base_img1.save("原稿/目次/原稿_001.png", dpi=(dpi, dpi))
  base_img2.save("原稿/目次/原稿_002.png", dpi=(dpi, dpi))
  print("目次の土台画像　読み込み完了")

def table_box_paste(author: Author):
  global mokuji_make
  if not mokuji_make:
    return False
  if author.manuscript_type =="その他":
    return False
  is_file_name = True
  is_file_title = True
  left_right = None #左か右か

  try:
    title_img = Image.open(f"原稿/目次/素材/{author.name}.png")
  except:
    is_file_title = False

  if not (is_file_title):
    return False

  if author.sequence: # 目次1ページ目に貼り付けか、2ページ目に貼り付けか
    if author.sequence < 9:
      left_right = 1
      y_num = author.sequence-1
    else:
      left_right = 2
      y_num = author.sequence-9

    base_img =  Image.open(f"原稿/目次/原稿_00{left_right}.png")

    # ペンネームとタイトルの貼り付け　ここを書き換えてください
    if is_file_title:
      if left_right == 1:
        base_img.paste(title_img, (mokuji_box_position[0], mokuji_box_position[1]+y_num*1260), title_img)
      else:
        base_img.paste(title_img, (manuscript_size[0]-mokuji_box_position[0]-3048-50, mokuji_box_position[1]+y_num*1260), title_img)
    base_img.save(f"原稿/目次/原稿_00{left_right}.png", dpi=(dpi, dpi))

def table_page_paste(sequence, page):
  if not mokuji_make:
    return False
  str_num = str(page)
  left_right = None #左か右か
  start_x = 300 #外側からの距離
  if sequence:
    if sequence < 9:
      left_right = 1
      y_num = sequence-1
    else:
      left_right = 2
      y_num = sequence-9

    base_img =  Image.open(f"原稿/目次/原稿_00{left_right}.png")
    glyphs = table_numbers
    w_list = table_number_widths
    # ノンブル、外側をそろえる
    if left_right == 1:
      total_w = sum(w_list[int(ch)] for ch in str_num)
      start_x = manuscript_size[0]-start_x-total_w
    cx = start_x
    for digit, ch in enumerate(str_num):
        d = glyphs[int(ch)]
        base_img.paste(d, (cx, mokuji_box_position[1]+570+y_num*1260), mask=d)  # LAでもRGBAでもOK（αがマスクになる）
        cx += w_list[int(ch)]

    base_img.save(f"原稿/目次/原稿_00{left_right}.png", dpi=(dpi, dpi))

def table_save():
  for i in range(2):
    table_img =  Image.open(f"原稿/目次/原稿_00{i+1}.png")
    table_img.save(f"原稿/全ての原稿/{(i+2):03d}.png", dpi=(dpi, dpi))

### アンソロジーを管理するclass

In [ ]:
# 最初の準備。ノンブルの画像など必要な画像などを読み込む
def Preparation():
  if has_nombre_book:
    ManuscriptPage.load_nombre_images()
    ManuscriptPage.make_white_image()
  if mokuji_make:
    table_reset()
  NonePage.make_white_image()
  print("-"*30)

# スプレッドシートからメンバーリスト、掲載順などの情報を持ってくる関数
def get_member_list():
  # スプレッドシートを開く
  try:
    ss = gc.open("執筆者名簿")
    st = ss.worksheet("名簿")
  except:
    print("spleadsheet 執筆者名簿が見つかりません")
    return
  # 中身取得
  member_list = st.get_all_values()[1:]
  print("執筆者リスト　読み込み完了\n"+ "-"*30)
  return member_list
  # [管理用番号, ペンネーム, 参加形態, 予定ページ数]　全てstr。注意

# 執筆者を一人一人原稿をチェックしていく関数

def get_check_authors(member_list: list):
  global global_page
  authors = {}
  for sequence, member in enumerate(member_list):
    if member[1] != "":
      if member[2] in ("イラスト", "漫画", "小説"):
        authors[member[1]] = Author(member[1], member[2], int(member[3]), global_page, sequence, folder=folder_path + f"/原稿/{member[1]}様")
        table_page_paste(sequence, global_page)
      else:
        authors[member[1]] = Author(member[1], member[2], int(member[3]), global_page, sequence, folder=folder_path + f"/原稿/{member[1]}")
      print(f"【 {authors[member[1]].name} 】 {authors[member[1]].folder}")
      global_page = authors[member[1]].author_check(global_page)
      table_box_paste(authors[member[1]])
  print(f"合計: {global_page-1}ページ")
  return authors

# 主催向けのプレビューを作成する関数
def make_preview():
    if not os.path.exists(f"{folder_path}/原稿/全ての原稿/プレビュー"):
      os.mkdir(f"{folder_path}/原稿/全ての原稿/プレビュー")

    manuscripts = natsorted([str(path) for path in list((pathlib.Path(f"{folder_path}/原稿/全ての原稿")).glob('???.png'))])
    if len(manuscripts)==0:# 1枚も原稿ファイルがなかった場合
      print("原稿がありません")
      return

    print("プレビュー作成開始: ", end="")
    if color_type:
        dodai = Image.new("L", (book_size[0]*2, book_size[1]), color=(255))
    else:
        dodai = Image.new("RGB", (book_size[0]*2, book_size[1]), color=(255, 255, 255))
    for manu_path in manuscripts:
      m = re.match(r"^(?P<page>\d{3}).png$", manu_path.split("/")[-1])
      page = int(m.group("page"))
      im = Image.open(manu_path).crop(((manuscript_size[0] - book_size[0]) // 2,
                                                (manuscript_size[1] - book_size[1]) // 2,
                                                (manuscript_size[0] + book_size[0]) // 2,
                                                (manuscript_size[1] + book_size[1]) // 2))
      if page%2==1:
        if im:
          print("-", end="")
          dodai.paste(im, (0 if toji=="右綴じ" else book_size[0], 0))
        dodai.save(f"{folder_path}/原稿/全ての原稿/プレビュー/preview_{page-1 if page-1>0 else ""}_{page}.png", dpi=(dpi, dpi))
      else:
        if color_type:
          dodai = Image.new("L", (book_size[0]*2, book_size[1]), color=(255))
        else:
          dodai = Image.new("RGB", (book_size[0]*2, book_size[1]), color=(255, 255, 255))
        if im:
          print(" , ", end="")
          dodai.paste(im, (book_size[0] if toji=="右綴じ" else 0, 0))
      print(f"{page}", end="")
    if page%2==0:
      dodai.save(f"{folder_path}/原稿/全ての原稿/プレビュー/preview_{page}_.png", dpi=(dpi, dpi))
    print(" |完了")

In [ ]:
# すべての原稿をまとめてpdfに変換する関数。印刷所によっては必要
def png_to_pdf():
  logging.getLogger("img2pdf").setLevel(logging.ERROR)

  pdf_path = "提出.pdf"
  img_paths = natsorted(list(pathlib.Path(f"{folder_path}/原稿/全ての原稿").glob('*.png')))

  with open(pdf_path,"wb") as file:
    file.write(img2pdf.convert([str(img) for img in img_paths]))

  print("PDFファイルへの変換が成功しました")

# 実行

In [ ]:
global_page = 1

Preparation()

member_list = get_member_list()
authors = get_check_authors(member_list)

if mokuji_make:
  table_save()
make_preview()
png_to_pdf()

# memo

In [ ]:
# 告知用の画像を作る用です

Preparation()

member_list = get_member_list()

authors = {}
now_expected_page = 1
for sequence, member in enumerate(member_list):
      if member[2] in ("イラスト", "漫画", "小説"):
        authors[member[1]] = Author(member[1], member[2], int(member[3]), now_expected_page, sequence, folder=folder_path + f"/原稿/{member[1]}様")
        now_expected_page += authors[member[1]].expected_pages+1
      else:
        authors[member[1]] = Author(member[1], member[2], int(member[3]), now_expected_page, sequence, folder=folder_path + f"/原稿/{member[1]}")
        now_expected_page += authors[member[1]].expected_pages

In [ ]:
#個別でチェックを行う場合、こちらを利用してください
check_name = "花子"


print(f"【 {authors[check_name].name} 】 {authors[check_name].folder}")
authors[check_name].author_check(authors[check_name].expected_start_page)
make_preview()
png_to_pdf()

In [ ]:
#目次のみ書き換え
table_reset()
for author in authors:
  print(author)
  if authors[author].manuscript_type in ("イラスト", "漫画", "小説"):
    table_page_paste(authors[author].sequence, authors[author].expected_start_page)
    table_box_paste(authors[author])
table_save()